# Phase 2 (projections variant) — Grid extension vs. standalone solar (share_dispersion)

Copy of `analyse_GIS_phase2/phase2_share_dispersion.ipynb`. Two changes from the original:

1. **PV/battery price & lifetime** (Section 1): `PV_EUR_KW`, `LIFE_PV`, `BAT_EUR_KWH` now come
   directly from the EnergyScope catalogue `reg_technologies.dat` (cross-checked against 2024
   retail prices, World Bank IDTR) instead of the Peña Balderrama et al. 2020 USD figures. `MV_EUR`,
   `CONN_EUR`, `OM_RATE`, `I_RATE`, and all SHS sizing assumptions (`BATT_DAYS`, `BATT_DOD`,
   `SYS_EFF`) are unchanged, as is the 2012 community geography and the ~1169 kWh/HH sizing.
2. **Section 7** (appended): re-runs the breakeven classification for three household scenarios —
   2024 (control, actual household counts, isolates the price/lifetime effect alone), 2035, and
   2050 (projected household counts by cluster from `projections/output/menages_projetes.csv`).

For every rural community in the study region we compare two ways to bring it electricity:
extending the medium-voltage (MV) grid, or installing a standalone solar home system (SHS).
Whichever is annually cheaper wins. The HH-weighted fraction of communities where the SHS wins,
per cluster, is `share_dispersion`.

Steps: parameters → per-municipality demand → SHS sizing → off-grid households already
electrified (Source B) → breakeven per community (Source B+C combined) → cluster aggregation →
projected household scenarios (2024/2035/2050).


In [1]:
import os
import pandas as pd

BASE = os.path.dirname(os.getcwd())            # repo root (notebook lives in analyse_GIS_phase2/)
OUT = "output"
os.makedirs(OUT, exist_ok=True)

# Cluster -> municipality membership (same mapping as analyse data ramp/demande.ipynb)
CLUSTERS = {
    "C1": ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    "C2": ["Bolpebra"],
    "C3": ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    "C4": ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza",
           "Porvenir", "Puerto_Rico", "San_Lorenzo", "San_Pedro",
           "Santa_Rosa_Pando", "Santos_Mercado", "Sena", "Villa_Nueva"],
    "C5": ["Cobija"],
}
MUNI_TO_CLUSTER = {m: c for c, munis in CLUSTERS.items() for m in munis}


## 1. Parameters

Cost data is from **Peña Balderrama et al. 2020, *Energy for Sustainable Development* 56**,
Table A.5 (grid extension) and Table A.7 (standalone SHS). FX and discount rate are fixed
model-wide conventions.


In [2]:
FX = 0.924          # 1 USD = 0.924 EUR — 2024 annual average (IRS/ECB)
I_RATE = 0.10        # model discount rate — Misc_indep.json

# Raw USD costs — Peña Balderrama et al. 2020, ESD 56 (MV/connection unchanged)
MV_USD = 9_000        # USD/km, MV 33kV extension — Table A.5
CONN_USD = 150        # USD/HH, last-mile connection — Table A.5
OM_RATE = 0.02        # %/yr O&M on overnight capex — Tables A.5/A.7
LIFE_MV = 30           # years — Table A.5

MV_EUR = MV_USD * FX
CONN_EUR = CONN_USD * FX

# PV and battery capex/lifetime — EnergyScope catalogue reg_technologies.dat, cross-checked
# against 2024 retail prices (World Bank IDTR). Replaces the Peña Balderrama et al. 2020 USD
# figures (Table A.7) used for these two parameters elsewhere in analyse_GIS_phase2/.
PV_EUR_KW = 2_730     # EUR/kW, standalone PV — reg_technologies.dat:10, retail 2024 x-checked IDTR World Bank
LIFE_PV = 20          # years — reg_technologies.dat:10, same source
BAT_EUR_KWH = 594     # EUR/kWh, battery — reg_technologies.dat:138
LIFE_BAT = 10          # years — unchanged

# SHS sizing assumptions — standard tropical off-grid design
SYS_EFF = 0.75       # PV -> usable AC, incl. battery/inverter/wiring losses
BATT_DAYS = 2        # days of autonomy
BATT_DOD = 0.80       # depth of discharge


def crf(r, n):
    """Capital recovery factor: annualizes an overnight cost over n years at rate r."""
    return r * (1 + r) ** n / ((1 + r) ** n - 1)


CRF_MV = crf(I_RATE, LIFE_MV)
CRF_PV = crf(I_RATE, LIFE_PV)
CRF_BAT = crf(I_RATE, LIFE_BAT)

pd.DataFrame([
    {"Parameter": "FX (USD->EUR)", "Value": FX, "Source": "2024 annual avg, IRS/ECB"},
    {"Parameter": "i_rate", "Value": I_RATE, "Source": "Misc_indep.json"},
    {"Parameter": "MV extension", "Value": f"{MV_USD} USD/km -> {MV_EUR:.0f} EUR/km", "Source": "Peña Balderrama 2020, Table A.5"},
    {"Parameter": "Connection", "Value": f"{CONN_USD} USD/HH -> {CONN_EUR:.1f} EUR/HH", "Source": "Peña Balderrama 2020, Table A.5"},
    {"Parameter": "PV capex", "Value": f"{PV_EUR_KW} EUR/kW", "Source": "reg_technologies.dat:10, retail 2024 x-checked IDTR World Bank"},
    {"Parameter": "Battery capex", "Value": f"{BAT_EUR_KWH} EUR/kWh", "Source": "reg_technologies.dat:138"},
    {"Parameter": "O&M rate", "Value": OM_RATE, "Source": "Peña Balderrama 2020, Tables A.5/A.7"},
    {"Parameter": "Lifetime MV/PV/Bat", "Value": f"{LIFE_MV}/{LIFE_PV}/{LIFE_BAT} y", "Source": "Table A.5 (MV) / reg_technologies.dat (PV, Bat)"},
    {"Parameter": "System efficiency", "Value": SYS_EFF, "Source": "off-grid SHS design assumption"},
    {"Parameter": "Battery autonomy", "Value": f"{BATT_DAYS} days", "Source": "off-grid SHS design assumption"},
    {"Parameter": "Depth of discharge", "Value": BATT_DOD, "Source": "off-grid SHS design assumption"},
    {"Parameter": "CRF_MV/PV/Bat", "Value": f"{CRF_MV:.5f}/{CRF_PV:.5f}/{CRF_BAT:.5f}", "Source": f"CRF(i={I_RATE}, n)"},
])


,Parameter,Value,Source
0,FX (USD->EUR),0.924,"2024 annual avg, IRS/ECB"
1,i_rate,0.1,Misc_indep.json
2,MV extension,9000 USD/km -> 8316 EUR/km,"Peña Balderrama 2020, Table A.5"
3,Connection,150 USD/HH -> 138.6 EUR/HH,"Peña Balderrama 2020, Table A.5"
4,PV capex,2730 EUR/kW,"reg_technologies.dat:10, retail 2024 x-checked..."
5,Battery capex,594 EUR/kWh,reg_technologies.dat:138
6,O&M rate,0.02,"Peña Balderrama 2020, Tables A.5/A.7"
7,Lifetime MV/PV/Bat,30/20/10 y,"Table A.5 (MV) / reg_technologies.dat (PV, Bat)"
8,System efficiency,0.75,off-grid SHS design assumption
9,Battery autonomy,2 days,off-grid SHS design assumption


## 2. Per-municipality demand

Annual electricity per household, `per_hh_kWh = sum(all sufficiency_* columns except
sufficiency_water_heating) / total_hh + ECS_APPOINT_KWH_HH`.

- **RAMP load curves**: `analyse data ramp/sufficiency/data ramp/<municipality>/load_curve_energy_service_full_year_Norte_Amazonia.csv`
  (power in W at 1-minute steps over a year → kWh = W·min / 60,000)
- **Total households**: `exctraction of data/output/CSV_final.csv`, Bolivia Census 2024 column
  `NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Total`

**`sufficiency_water_heating` is excluded** from the electric SHS sizing: dispersed/off-grid
households do not run electric water heating in reality (thermosiphon, direct solar, or no hot
water at all), so including it would over-size the standalone PV+battery system and understate
`share_dispersion`. Only the small electric-resistance fraction found in the "appoint réaliste"
sufficiency variant (`EnergyScope_BO_nord_amazonia`, DEC_HP_ELEC/TS_DEC_HP_ELEC f_max=0, DEC_SOLAR
backed by DEC_DIRECT_ELEC as host tech) is added back, via `ECS_APPOINT_KWH_HH` below.

The RAMP "sufficiency" scenario is uniform by design, so every municipality should land close to
the same per-HH value — used here as a sanity check, not just a calculation.


In [3]:
csv_final = pd.read_csv(
    os.path.join(BASE, "exctraction of data/output/CSV_final.csv"), encoding="utf-8-sig"
)
hh_col = [c for c in csv_final.columns if "2024" in c and "Total" in c and "VIVIENDA" in c.upper()][0]


def std_muni_name(depto, raw):
    """Match CSV_final's municipality names to the CLUSTERS keys.
    'Santa Rosa' exists in both Beni and Pando — disambiguate by department."""
    raw = str(raw).strip()
    if raw == "Santa Rosa":
        return "Santa_Rosa_Beni" if depto == "Beni" else "Santa_Rosa_Pando"
    return raw.replace(" ", "_")


muni_rows = csv_final[csv_final["MUNICIPIO/TIOC"].notna()]
total_hh = {
    std_muni_name(row["DEPARTAMENTO"], row["MUNICIPIO/TIOC"]): row[hh_col]
    for _, row in muni_rows.iterrows()
    if std_muni_name(row["DEPARTAMENTO"], row["MUNICIPIO/TIOC"]) in MUNI_TO_CLUSTER
}
assert set(total_hh) == set(MUNI_TO_CLUSTER), "missing total_hh for some municipalities"


In [4]:
RAMP_DIR = os.path.join(BASE, "analyse data ramp/sufficiency/data ramp")

# APPOINT_ELEC_SHARE: region-wide fraction of decentralised hot-water (ECS / HEAT_LOW_T_DECEN)
# service met by the electric resistance host tech (DEC_DIRECT_ELEC) in the "appoint réaliste"
# sufficiency variant of EnergyScope_BO_nord_amazonia (Data/2025/sufficiency/C{1-5}/Technologies.csv:
# DEC_HP_ELEC and TS_DEC_HP_ELEC f_max=0, so DEC_SOLAR is backed by DEC_DIRECT_ELEC as host tech).
# Source: sufficiency_phase2 run (2026-07-19/20), region-wide DEC_DIRECT_ELEC HEAT_LOW_T_DECEN
# production / total "Hot water" demand = 3.9468 / 71.6685 GWh = 0.04825.
APPOINT_ELEC_SHARE = 0.04825
ECS_APPOINT_KWH_HH = 851 * APPOINT_ELEC_SHARE  # kWh/HH/yr added back as electric appoint

per_hh_kwh = {}
no_ramp_file = []

for muni in MUNI_TO_CLUSTER:
    path = os.path.join(RAMP_DIR, muni, "load_curve_energy_service_full_year_Norte_Amazonia.csv")
    if not os.path.exists(path):
        no_ramp_file.append(muni)
        continue
    df = pd.read_csv(path)
    # sufficiency_water_heating excluded on purpose (see section 2 markdown): dispersed/off-grid
    # households do not run electric water heating in reality. Only ECS_APPOINT_KWH_HH (the small
    # electric-resistance fraction from étape A) is added back below, uniformly per household.
    suff_cols = [c for c in df.columns if c.startswith("sufficiency_") and c != "sufficiency_water_heating"]
    kwh_total = df[suff_cols].sum().sum() / 60_000.0
    per_hh_kwh[muni] = kwh_total / total_hh[muni] + ECS_APPOINT_KWH_HH

# A few municipalities (e.g. Guayaramerín) have no RAMP run -> use the
# total_hh-weighted average of their cluster peers instead.
for muni in no_ramp_file:
    cluster = MUNI_TO_CLUSTER[muni]
    peers = [p for p in CLUSTERS[cluster] if p in per_hh_kwh]
    per_hh_kwh[muni] = (sum(per_hh_kwh[p] * total_hh[p] for p in peers)
                        / sum(total_hh[p] for p in peers))
    print(f"{muni}: no RAMP file -> fallback to {cluster} average over {peers}")

demand = pd.DataFrame([
    {"Cluster": MUNI_TO_CLUSTER[m], "Municipio": m, "total_hh": total_hh[m],
     "per_hh_kWh": round(per_hh_kwh[m], 1)}
    for m in MUNI_TO_CLUSTER
]).sort_values(["Cluster", "Municipio"])

mean_kwh = demand["per_hh_kWh"].mean()
demand["pct_dev_from_mean"] = (demand["per_hh_kWh"] / mean_kwh - 1) * 100
print(f"mean per_hh_kWh = {mean_kwh:.1f}  (sufficiency scenario is uniform by design; "
      f"ECS_APPOINT_KWH_HH = {ECS_APPOINT_KWH_HH:.2f} kWh/HH/yr added to every municipality)")
demand


mean per_hh_kWh = 1168.9  (sufficiency scenario is uniform by design; ECS_APPOINT_KWH_HH = 41.06 kWh/HH/yr added to every municipality)


,Cluster,Municipio,total_hh,per_hh_kWh,pct_dev_from_mean
0,C1,Exaltación,1455.0,1173.8,0.416743
3,C1,Ixiamas,3306.0,1131.4,-3.210510
1,C1,Reyes,3417.0,1158.1,-0.926367
2,C1,Santa_Rosa_Beni,2755.0,1176.3,0.630614
4,C2,Bolpebra,802.0,1166.8,-0.182096
5,C3,Guayaramerín,10891.0,1175.3,0.545066
7,C3,Puerto_Gonzalo_Moreno,1965.0,1173.4,0.382524
6,C3,Riberalta,27442.0,1174.9,0.510846
8,C4,Bella_Flor,1235.0,1169.4,0.040330
9,C4,Filadelfia,2514.0,1164.3,-0.395967


In [5]:
flagged = demand[demand["pct_dev_from_mean"].abs() > 5]
if len(flagged):
    print("FLAGGED — more than 5% off the mean:")
    print(flagged)
else:
    print("All municipalities within 5% of the mean — sanity check passed.")


All municipalities within 5% of the mean — sanity check passed.


## 3. Standalone SHS sizing

Each household's PV + battery system is sized to cover its own municipality's demand:

- PV size: `kWh/yr / (CF × 8760 × η_system)`, with `η_system = 0.75`
- Battery size: `(2 days autonomy × daily kWh) / depth_of_discharge`
- Solar capacity factor (CF) per cluster: mean of the `PV` column in
  `analyse data ramp/sufficiency/output_energyscope/<cluster>/Time_series.csv` (EnergyScope's own solar profile)


**Note — no thermosiphon/solar-water-heater capex added to the breakeven.** ECS (hot water) is
served in this model by `DEC_SOLAR` backed by `DEC_DIRECT_ELEC` as host tech (see étape A) — the
real-world equivalent is a solar thermosiphon with a small electric-resistance backup. This
equipment is needed by households on **both sides** of the breakeven comparison (grid-connected
and dispersed/SHS alike), so its cost cancels out and is deliberately **not** added to either the
`extension` or `standalone` annual cost in `classify_communities`. Only the electric-resistance
kWh (`ECS_APPOINT_KWH_HH`, added to `per_hh_kwh` above) affects PV+battery sizing, since that
portion is genuinely electric.


In [6]:
cf_pv = {}
for cluster in CLUSTERS:
    ts = pd.read_csv(
        os.path.join(BASE, f"analyse data ramp/sufficiency/output_energyscope/{cluster}/Time_series.csv"),
        sep=";", index_col=0,
    )
    cf_pv[cluster] = ts["PV"].mean()


def size_shs(kwh_per_year, cf, crf_pv=CRF_PV, crf_bat=CRF_BAT):
    """Return (overnight capex, annualized cost) in EUR per household."""
    pv_kw = kwh_per_year / (cf * 8760 * SYS_EFF)
    bat_kwh = (BATT_DAYS * kwh_per_year / 365) / BATT_DOD
    pv_eur = pv_kw * PV_EUR_KW
    bat_eur = bat_kwh * BAT_EUR_KWH
    capex = pv_eur + bat_eur
    annual = crf_pv * pv_eur + crf_bat * bat_eur + OM_RATE * capex
    return capex, annual


shs_capex, shs_annual = {}, {}
for muni, kwh in per_hh_kwh.items():
    shs_capex[muni], shs_annual[muni] = size_shs(kwh, cf_pv[MUNI_TO_CLUSTER[muni]])

pd.DataFrame([
    {"Cluster": MUNI_TO_CLUSTER[m], "Municipio": m,
     "overnight_capex_EUR_HH": round(shs_capex[m], 0),
     "annual_EUR_HH": round(shs_annual[m], 2)}
    for m in MUNI_TO_CLUSTER
]).sort_values(["Cluster", "Municipio"])


,Cluster,Municipio,overnight_capex_EUR_HH,annual_EUR_HH
0,C1,Exaltación,7580.0,1258.21
3,C1,Ixiamas,7306.0,1212.80
1,C1,Reyes,7479.0,1241.38
2,C1,Santa_Rosa_Beni,7596.0,1260.89
4,C2,Bolpebra,7582.0,1257.14
5,C3,Guayaramerín,7576.0,1257.87
7,C3,Puerto_Gonzalo_Moreno,7563.0,1255.82
6,C3,Riberalta,7573.0,1257.49
8,C4,Bella_Flor,7712.0,1275.52
9,C4,Filadelfia,7678.0,1269.91


## 4. Breakeven per community

For each community, compare the annualized cost of extending the grid against the annualized SHS
cost. Community-level inputs (distance to nearest line, household count) come from
`analyse_GIS_phase2/output/task1_community_distances_lines.csv`, built upstream in
`gis_phase2_analysis.ipynb`. Household count used in the classification below is Source C
(unelectrified, no-tiene 2012 scaled to 2024) **plus** Source B (already off-grid electrified —
counted in section 5): they live in the same communities and share the same line/distance, so the
connection decision reflects everyone who would benefit from it.

$$\text{annual\_extension} = (CRF_{MV} + OM) \times MV_{EUR/km} \times \frac{dist}{HH} + (CRF_{MV} + OM) \times CONN_{EUR}$$

A community is **dispersed** (SHS wins) if its annual SHS cost is lower than this. The breakeven
distance `d*` is where the two costs are equal — it scales with household count.


In [7]:
task1 = pd.read_csv(
    os.path.join(BASE, "analyse_GIS_phase2/output/task1_community_distances_lines.csv"),
    encoding="utf-8-sig",
)


def classify_communities(communities, shs_annual_by_muni, mv_eur_per_km, conn_eur, crf_mv):
    rows = []
    for _, row in communities.iterrows():
        muni, dist, hh = row["Municipio_std"], row["distance_km"], row["HH_2024_scaled"]
        if hh <= 0:
            continue
        standalone = shs_annual_by_muni[muni]
        slope = (crf_mv + OM_RATE) * mv_eur_per_km
        extension = slope * dist / hh + (crf_mv + OM_RATE) * conn_eur
        d_star = max((standalone - (crf_mv + OM_RATE) * conn_eur) * hh / slope, 0.0)
        rows.append({
            "Cluster": row["Cluster"], "Comunidad": row["Comunidad"], "Municipio_std": muni,
            "HH": hh, "distance_km": dist, "nearest_line_type": row["nearest_line_type"],
            "annual_extension_EUR_HH": round(extension, 2),
            "annual_standalone_EUR_HH": round(standalone, 2),
            "dispersed": standalone < extension,
            "breakeven_distance_km": round(d_star, 3),
        })
    return pd.DataFrame(rows)


## 5. Off-grid households already electrified (Source B)

Source B households are already electrified off-grid in 2012 (diesel generator, solar panel, or
other). They live in the **same communities** as Source C, so the distance and line infrastructure
are shared. Adding them to each community increases its household count, which lowers the per-HH
line cost and can make a community connectable that Source C alone would not justify.

Sources:
- **2012 community counts**: `analyse_GIS_phase2/data/comunidades_electricidad_2012.csv`,
  columns `Motor`, `Panel`, `Otra` (capped at `Hog_Elec - Red_elec` to avoid double-counting)
- **2024 municipal totals**: `exctraction of data/output/CSV_final.csv`, 2024 Motor/Panel/Otra columns


In [8]:
# Home-system unit sizes — aligned with analyse data ramp/reality/home_systems.ipynb
# Source: ENDE BO-L1222 / Programa de Electrificación Rural
PANEL_W = 50                   # Wp per panel-HH (ENDE PEVD kit: >=50 Wp policristalino)
GEN_W = 500                    # W per generator-HH (assumption, no field data — sensitivity at 350 W)
BATT_KWH_PER_PANEL_HH = 0.123  # kWh per panel-HH (ENDE PEVD kit: 123 Wh lithium 12V)


In [9]:
# ── 2024 municipal B totals from CSV_final ────────────────────────────────────
motor_col = [c for c in csv_final.columns if "2024" in c and "Motor" in c and "VIVIENDA" in c.upper()][0]
panel_col = [c for c in csv_final.columns if "2024" in c and "Panel" in c][0]
otra_col  = [c for c in csv_final.columns if "2024" in c and "Otra"  in c][0]
print("Source B columns from CSV_final:")
print(f"  motor: '{motor_col}'")
print(f"  panel: '{panel_col}'")
print(f"  otra:  '{otra_col}'")

muni_b = {}   # muni -> {total, motor, panel, otra}
for _, row in csv_final[csv_final["MUNICIPIO/TIOC"].notna()].iterrows():
    name = std_muni_name(row["DEPARTAMENTO"], row["MUNICIPIO/TIOC"])
    if name in MUNI_TO_CLUSTER:
        muni_b[name] = {
            "total": row[motor_col] + row[panel_col] + row[otra_col],
            "motor": row[motor_col],
            "panel": row[panel_col],
            "otra":  row[otra_col],
        }

b_check = pd.DataFrame([
    {"Cluster": MUNI_TO_CLUSTER[m], "Municipio": m,
     "Motor_2024": muni_b[m]["motor"], "Panel_2024": muni_b[m]["panel"],
     "Otra_2024": muni_b[m]["otra"], "B_total_2024": muni_b[m]["total"]}
    for m in MUNI_TO_CLUSTER
]).sort_values(["Cluster", "Municipio"])
print(f"\nTotal B households across all municipalities: {b_check['B_total_2024'].sum():.0f}")
b_check


Source B columns from CSV_final:
  motor: 'NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Motor propio'
  panel: 'NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Panel solar'
  otra:  'NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Otra'

Total B households across all municipalities: 9325


,Cluster,Municipio,Motor_2024,Panel_2024,Otra_2024,B_total_2024
0,C1,Exaltación,203.0,664.0,23.0,890.0
3,C1,Ixiamas,337.0,353.0,103.0,793.0
1,C1,Reyes,96.0,213.0,44.0,353.0
2,C1,Santa_Rosa_Beni,198.0,405.0,17.0,620.0
4,C2,Bolpebra,160.0,216.0,17.0,393.0
5,C3,Guayaramerín,166.0,383.0,97.0,646.0
7,C3,Puerto_Gonzalo_Moreno,48.0,58.0,126.0,232.0
6,C3,Riberalta,422.0,692.0,282.0,1396.0
8,C4,Bella_Flor,67.0,160.0,88.0,315.0
9,C4,Filadelfia,334.0,240.0,34.0,608.0


In [10]:
# ── Load 2012 community-level electrification survey ─────────────────────────
# Columns used below: Comunidad, Motor/Panel/Otra (off-grid counts), Hog_Elec, Red_elec.
# Note: this file encodes one municipality as "Tercera Sección - Santa Rosa" (Depto = Beni), which
# the name-extraction regex turns into "Santa Rosa" — disambiguated below by department so it
# joins correctly with task1.
com2012 = pd.read_csv(
    os.path.join(BASE, "analyse_GIS_phase2/data/comunidades_electricidad_2012.csv"),
    encoding="utf-8-sig", low_memory=False,
)
com2012["Municipio_std"] = com2012["Municipio"].str.extract(r"- (.+)$")[0].str.strip()
com2012["Municipio_std"] = com2012["Municipio_std"].fillna(com2012["Municipio"].str.strip())

# "Santa Rosa" is ambiguous (exists in both Beni and Pando) — disambiguate by department BEFORE
# the general space->underscore normalization below, since "Santa_Rosa" alone isn't a valid
# MUNI_TO_CLUSTER key. Substring match (not exact): Pando's community is recorded as "Santa Rosa
# del Abuná", not literally "Santa Rosa" — same substring gis_phase2_analysis.ipynb's own
# MUNI_FILTER (cell 5) already uses for this municipality, so this isn't a new assumption.
is_santa_rosa = com2012["Municipio_std"].str.contains("Santa Rosa", na=False)
com2012.loc[is_santa_rosa & (com2012["Depto"] == "Beni"), "Municipio_std"] = "Santa_Rosa_Beni"
com2012.loc[is_santa_rosa & (com2012["Depto"] == "Pando"), "Municipio_std"] = "Santa_Rosa_Pando"
print(f"renamed {is_santa_rosa.sum()} 'Santa Rosa*' rows by department "
      f"({(com2012['Municipio_std'] == 'Santa_Rosa_Beni').sum()} -> Santa_Rosa_Beni, "
      f"{(com2012['Municipio_std'] == 'Santa_Rosa_Pando').sum()} -> Santa_Rosa_Pando)")

# Villa Nueva: raw name carries a parenthetical suffix, "Villa Nueva (Loma Alta)", not stripped by
# the general space->underscore replace below. Single case, no department ambiguity (only Pando).
is_villa_nueva = com2012["Municipio_std"].str.contains("Villa Nueva", na=False)
com2012.loc[is_villa_nueva, "Municipio_std"] = "Villa_Nueva"
print(f"renamed {is_villa_nueva.sum()} 'Villa Nueva*' rows -> Villa_Nueva")

# General normalization: MUNI_TO_CLUSTER and task1's own Municipio_std use underscores for every
# multi-word name (e.g. "Puerto_Gonzalo_Moreno"), not just the two cases above. The previous ad
# hoc single-case patch left every other multi-word municipality as space-separated, so it was
# silently dropped by the isin() filter below — no error, no warning.
com2012["Municipio_std"] = com2012["Municipio_std"].str.replace(" ", "_", regex=False)

# Blocking assertion: every municipality in MUNI_TO_CLUSTER must have at least one matching row
# here, or Source B for it silently resolves to 0 downstream (via cell 17's reindex().fillna(0)).
present = set(com2012["Municipio_std"].unique())
missing = set(MUNI_TO_CLUSTER) - present
assert not missing, f"No com2012 rows match these MUNI_TO_CLUSTER municipalities: {sorted(missing)}"
print("All MUNI_TO_CLUSTER municipalities have a match in com2012 -> OK")

# ── 2012 community-level B counts → scale to 2024 ──────────────────────────
# B_2012 = min(Motor+Panel+Otra, Hog_Elec - Red_elec) to avoid double-counting
study_b = com2012[com2012["Municipio_std"].isin(MUNI_TO_CLUSTER)].copy()
study_b["B_2012_raw"] = (study_b["Motor"].fillna(0)
                         + study_b["Panel"].fillna(0)
                         + study_b["Otra"].fillna(0))
study_b["B_cap"]  = (study_b["Hog_Elec"].fillna(0) - study_b["Red_elec"].fillna(0)).clip(lower=0)
study_b["B_2012"] = study_b[["B_2012_raw", "B_cap"]].min(axis=1)
study_b["Panel_2012"] = study_b["Panel"].fillna(0)
study_b["Motor_2012"] = study_b["Motor"].fillna(0)
study_b["Otra_2012"]  = study_b["Otra"].fillna(0)
study_b["Comunidad_norm"] = study_b["Comunidad"].str.strip().str.upper()

# Scale each community's 2012 count proportionally to the 2024 municipal total
# (same method used for Source C scaling to 2024). Panel/Motor/Otra scale to their
# own 2024 municipal totals; "Otra" is kept separate here and split 50/50 later.
for col_2012, b_key in [("B_2012", "total"), ("Panel_2012", "panel"),
                        ("Motor_2012", "motor"), ("Otra_2012", "otra")]:
    col_2024 = col_2012.replace("_2012", "_2024")
    denom = study_b.groupby("Municipio_std")[col_2012].transform("sum")
    b24   = study_b["Municipio_std"].map({m: muni_b[m][b_key] for m in muni_b})
    study_b[col_2024] = (study_b[col_2012] / denom.replace(0, float("nan")) * b24).fillna(0)

# Aggregate to (Municipio_std, Comunidad_norm) — removes any duplicate rows
b_comm = (
    study_b.groupby(["Municipio_std", "Comunidad_norm"])[["B_2024", "Panel_2024", "Motor_2024", "Otra_2024"]]
    .sum()
    .reset_index()
)
print(f"Communities with B data: {len(b_comm)} (out of {len(task1)} in task1)")

renamed 127 'Santa Rosa*' rows by department (29 -> Santa_Rosa_Beni, 15 -> Santa_Rosa_Pando)
renamed 21 'Villa Nueva*' rows -> Villa_Nueva
All MUNI_TO_CLUSTER municipalities have a match in com2012 -> OK
Communities with B data: 706 (out of 697 in task1)


In [11]:
# ── Assign B values onto task1 without merging (avoids row duplication) ───
# Some community names repeat within a municipality (e.g. "San Pedro" in Bella_Flor).
# Indexing by (Municipio_std, Comunidad_norm) preserves the exact 697-row structure of task1.
BCOLS = ["B_2024", "Panel_2024", "Motor_2024", "Otra_2024"]
b_indexed = b_comm.set_index(["Municipio_std", "Comunidad_norm"])[BCOLS]

task1_bc = task1.copy()
task1_bc["Comunidad_norm"] = task1_bc["Comunidad"].str.strip().str.upper()

# Join key: (Municipio_std, Comunidad_norm) -- unchanged from before. What changes is how the
# per-community b_comm value is applied when this key isn't unique within task1: previously
# .reindex() assigned the SAME (already-summed) value to every task1 row sharing a key, silently
# doubling (or tripling) B_2024 wherever two physically distinct communities share a name within
# one municipality (e.g. two "SAN ANTONIO" in Sena, three "PALESTINA" in Riberalta). Fixed by
# splitting the b_comm value across homonymous rows instead of duplicating it, weighted by each
# row's 2012 population (Poblacion_2012 -- task1's own 2012-vintage size weight, the natural
# choice for splitting a household count without an independent per-row 2012 household figure)
# when the group's total population is nonzero, or equally when it is not.
GROUP_KEYS = ["Municipio_std", "Comunidad_norm"]
group_pop = task1_bc.groupby(GROUP_KEYS)["Poblacion_2012"].transform("sum")
n_in_group = task1_bc.groupby(GROUP_KEYS)["Comunidad_norm"].transform("size")
share = (task1_bc["Poblacion_2012"] / group_pop).where(group_pop > 0, 1.0 / n_in_group)

key_index = pd.MultiIndex.from_frame(task1_bc[GROUP_KEYS])
b_per_key = b_indexed.reindex(key_index).fillna(0.0)
for col in BCOLS:
    task1_bc[col] = b_per_key[col].values * share.values

n_dup_rows = int((n_in_group > 1).sum())
n_dup_groups = int(task1_bc.loc[n_in_group > 1, GROUP_KEYS].drop_duplicates().shape[0])
print(f"Homonym groups (>1 community sharing the same name within a municipality): "
      f"{n_dup_groups}, rows involved: {n_dup_rows}")

matched_b = (task1_bc["B_2024"] > 0).sum()
print(f"Communities with B households matched: {matched_b}/{len(task1_bc)}")
total_assigned = task1_bc["B_2024"].sum()
print(f"Total B households assigned: {total_assigned:.4f}")
census_total = sum(muni_b[m]["total"] for m in muni_b)
print(f"  (municipal totals from CSV_final: {census_total:.0f})")

# Known residual: com2012 community names with no match in task1's GIS layer at all (pre-existing
# data-quality gap between two independently-compiled community lists -- distinct from the two
# defects fixed above, not addressed here). Accounted for explicitly rather than silently ignored,
# so the blocking assertion below catches any *unexplained* gap.
task1_keys_set = set(zip(task1_bc["Municipio_std"], task1_bc["Comunidad_norm"]))
b_comm_keys_set = set(zip(b_comm["Municipio_std"], b_comm["Comunidad_norm"]))
orphan_keys = b_comm_keys_set - task1_keys_set
orphan_total = (b_comm.set_index(["Municipio_std", "Comunidad_norm"]).loc[list(orphan_keys), "B_2024"].sum()
                if orphan_keys else 0.0)
print(f"  Known residual: {len(orphan_keys)} 2012-census community names have no match in task1 "
      f"-> {orphan_total:.1f} households unavoidably unassigned by this join (community-name "
      f"mismatch between 2012 census and GIS layer, not fixed here)")

# Blocking assertion: every household in the census total must be either assigned or accounted for
# by the known residual above -- i.e. no unexplained gap beyond the two fixed defects.
assert abs((total_assigned + orphan_total) - census_total) < 1.0, (
    f"assigned ({total_assigned:.1f}) + known residual ({orphan_total:.1f}) != census total "
    f"({census_total:.0f}): unexplained gap beyond the two fixed defects and the known "
    "community-name mismatch"
)
print(f"ASSERTION (assigned + known residual == census total {census_total:.0f}) -> OK")

# HH for breakeven = C (no-tiene scaled) + B — larger HH → lower per-HH extension cost,
# so a community with enough B households can be connectable even if C alone wouldn't be.
task1_bc["HH_2024_scaled"] = task1_bc["HH_2024_scaled"] + task1_bc["B_2024"]

communities_bc = classify_communities(task1_bc, shs_annual, MV_EUR, CONN_EUR, CRF_MV)

# Attach Panel/Motor/Otra for TECH_HS estimation.
# classify_communities skips rows with HH <= 0 and returns results in the same order —
# so we can align by position with the non-zero-HH rows of task1_bc.
task1_bc_valid = task1_bc[task1_bc["HH_2024_scaled"] > 0].reset_index(drop=True)
communities_bc["Panel_2024"] = task1_bc_valid["Panel_2024"].values
communities_bc["Motor_2024"] = task1_bc_valid["Motor_2024"].values
communities_bc["Otra_2024"]  = task1_bc_valid["Otra_2024"].values

print(f"{len(communities_bc)} communities: "
      f"{(~communities_bc['dispersed']).sum()} connectable, {communities_bc['dispersed'].sum()} dispersed")

Homonym groups (>1 community sharing the same name within a municipality): 20, rows involved: 41
Communities with B households matched: 604/697
Total B households assigned: 9078.0789
  (municipal totals from CSV_final: 9325)
  Known residual: 30 2012-census community names have no match in task1 -> 246.9 households unavoidably unassigned by this join (community-name mismatch between 2012 census and GIS layer, not fixed here)
ASSERTION (assigned + known residual == census total 9325) -> OK
697 communities: 444 connectable, 253 dispersed


## 6. Aggregate per cluster

`share_dispersion` is the fraction of ALL households (Source A + B + C, from CSV_final) in a
cluster that end up off-grid (SHS) rather than grid-connected.


In [12]:
def aggregate_clusters(comm_bc):
    # Denominator = total HH per cluster from CSV_final (Source A + B + C)
    rows = []
    for cluster in CLUSTERS:
        sub = comm_bc[comm_bc["Cluster"] == cluster]
        total_hh_csv  = sum(total_hh[m] for m in CLUSTERS[cluster])   # A+B+C denominator
        hh_dispersed  = sub.loc[sub["dispersed"], "HH"].sum()
        connectable   = ~sub["dispersed"]
        ext_line = (sub.loc[connectable, "distance_km"] * MV_EUR).sum()
        ext_conn = sub.loc[connectable, "HH"].sum() * CONN_EUR         # C+B connectable HH

        # Dispersed B: existing off-grid equipment fed to EnergyScope as brownfield f_min.
        # Unit sizes from ENDE (PANEL_W / GEN_W / BATT_KWH_PER_PANEL_HH), aligned with home_systems.ipynb.
        # "Otra" (unspecified off-grid source) split 50/50 between PV and diesel, same as home_systems.
        disp = sub["dispersed"]
        disp_panel = sub.loc[disp, "Panel_2024"].sum() + sub.loc[disp, "Otra_2024"].sum() / 2
        disp_motor = sub.loc[disp, "Motor_2024"].sum() + sub.loc[disp, "Otra_2024"].sum() / 2

        rows.append({
            "Cluster": cluster,
            "share_dispersion": round(hh_dispersed / total_hh_csv, 4) if total_hh_csv else 0.0,
            "extension_capex_EUR": round(ext_line + ext_conn, 0),
            "N_connectable": int(connectable.sum()),
            "N_dispersed": int(sub["dispersed"].sum()),
            # TECH_HS brownfield capacity for EnergyScope f_min (W -> GW, kWh -> GWh)
            "f_min_PV_HS_GW":     round(disp_panel * PANEL_W / 1e9, 8),
            "f_min_HS_DIESEL_GW": round(disp_motor * GEN_W / 1e9, 8),
            "f_min_BATT_HS_GWh":  round(disp_panel * BATT_KWH_PER_PANEL_HH / 1e6, 8),
        })
    return pd.DataFrame(rows)


summary_bc = aggregate_clusters(communities_bc)
summary_bc


,Cluster,share_dispersion,extension_capex_EUR,N_connectable,N_dispersed,f_min_PV_HS_GW,f_min_HS_DIESEL_GW,f_min_BATT_HS_GWh
0,C1,0.1660,9168870.0,76,88,2.509000e-05,0.000187,0.000062
1,C2,0.1487,1614644.0,10,6,6.500000e-07,0.000021,0.000002
2,C3,0.0179,7566979.0,117,43,8.070000e-06,0.000058,0.000020
3,C4,0.0774,19370468.0,227,116,1.043000e-05,0.000153,0.000026
4,C5,0.0000,157242.0,14,0,0.000000e+00,0.000000,0.000000


### Documentation — changement d'hypothèses & crédit de l'équipement existant (hors breakeven)

Deux vérifications gardées **en dehors** du breakeven, pour les limitations de la thèse :

1. **Avant/après** l'harmonisation des tailles unitaires avec `home_systems.ipynb`
   (panneau 80 W → 50 W, ajout batterie, `Otra` désormais splitée 50/50 PV/diesel).
2. **Valeur du kit existant de la Source B** en % du coût standalone qu'elle paierait sinon.
   Ce crédit n'est **volontairement pas** soustrait dans `classify_communities` — il est déjà porté par
   le `f_min` brownfield d'EnergyScope (PV_HS / BATT_HS). Affiché ici uniquement pour quantifier sa petitesse.


In [13]:
# ── (documentation) Impact of harmonizing home-system unit sizes with home_systems.ipynb ──
# OLD phase2 assumptions: panel 80 W, motor 500 W, no battery, "Otra" excluded
# NEW (this notebook):    panel 50 W (=0.625x), motor 500 W, battery 0.123 kWh/panel-HH, "Otra" split 50/50
OLD_PANEL_W, OLD_GEN_W = 80, 500
cmp_rows = []
for cluster in CLUSTERS:
    sub = communities_bc[communities_bc["Cluster"] == cluster]
    d = sub["dispersed"]
    panel_old = sub.loc[d, "Panel_2024"].sum()                     # no Otra
    motor_old = sub.loc[d, "Motor_2024"].sum()                     # no Otra
    panel_new = panel_old + sub.loc[d, "Otra_2024"].sum() / 2      # +Otra/2
    motor_new = motor_old + sub.loc[d, "Otra_2024"].sum() / 2      # +Otra/2
    cmp_rows.append({
        "Cluster": cluster,
        "PV_HS_MW_old(80W)":      round(panel_old * OLD_PANEL_W / 1e6, 4),
        "PV_HS_MW_new(50W+Otra)": round(panel_new * PANEL_W / 1e6, 4),
        "DIESEL_MW_old":          round(motor_old * OLD_GEN_W / 1e6, 4),
        "DIESEL_MW_new(+Otra)":   round(motor_new * GEN_W / 1e6, 4),
    })
fmin_cmp = pd.DataFrame(cmp_rows)
print("f_min before/after harmonization (50/80 = 0.625 unit effect; the rest is the added Otra/2):")
display(fmin_cmp)

# ── (documentation, NOT applied to the breakeven) Value of B's EXISTING kit vs the standalone cost ──
# Kept out of classify_communities on purpose: this credit is already carried by EnergyScope's
# brownfield f_min (PV_HS / BATT_HS). Shown here only to quantify how small it is, for the thesis limitations.
kit_pv_capex  = (PANEL_W / 1000) * PV_EUR_KW            # EUR/panel-HH, overnight PV
kit_bat_capex = BATT_KWH_PER_PANEL_HH * BAT_EUR_KWH     # EUR/panel-HH, overnight battery
kit_annual = (CRF_PV * kit_pv_capex + CRF_BAT * kit_bat_capex
              + OM_RATE * (kit_pv_capex + kit_bat_capex))   # EUR/yr/panel-HH
print(f"\nExisting ENDE kit = {PANEL_W} Wp PV + {BATT_KWH_PER_PANEL_HH*1000:.0f} Wh battery "
      f"= {kit_pv_capex:.0f} + {kit_bat_capex:.0f} = {kit_pv_capex + kit_bat_capex:.0f} EUR overnight "
      f"-> {kit_annual:.1f} EUR/yr annualized")
credit_rows = []
for cluster in CLUSTERS:
    shs_cluster = sum(shs_annual[m] for m in CLUSTERS[cluster]) / len(CLUSTERS[cluster])  # mean standalone EUR/yr/HH
    credit_rows.append({
        "Cluster": cluster,
        "standalone_EUR_yr_HH":   round(shs_cluster, 1),
        "existing_kit_EUR_yr_HH": round(kit_annual, 1),
        "credit_if_applied_%":    round(100 * kit_annual / shs_cluster, 2),
    })
credit_doc = pd.DataFrame(credit_rows)
print("If B's existing kit were credited against the full-sufficiency standalone cost, it would lower it by:")
display(credit_doc)


f_min before/after harmonization (50/80 = 0.625 unit effect; the rest is the added Otra/2):


,Cluster,PV_HS_MW_old(80W),PV_HS_MW_new(50W+Otra),DIESEL_MW_old,DIESEL_MW_new(+Otra)
0,C1,0.0384,0.0251,0.1764,0.1870
1,C2,0.0010,0.0007,0.0206,0.0209
2,C3,0.0121,0.0081,0.0526,0.0580
3,C4,0.0143,0.0104,0.1381,0.1529
4,C5,0.0000,0.0000,0.0000,0.0000



Existing ENDE kit = 50 Wp PV + 123 Wh battery = 136 + 73 = 210 EUR overnight -> 32.1 EUR/yr annualized
If B's existing kit were credited against the full-sufficiency standalone cost, it would lower it by:


,Cluster,standalone_EUR_yr_HH,existing_kit_EUR_yr_HH,credit_if_applied_%
0,C1,1243.3,32.1,2.58
1,C2,1257.1,32.1,2.55
2,C3,1257.1,32.1,2.55
3,C4,1277.2,32.1,2.51
4,C5,1247.4,32.1,2.57


In [14]:
summary_bc.to_csv("output/share_dispersion_final_BC.csv", index=False)
communities_bc.to_csv("output/community_breakeven_detail_BC.csv", index=False)
print("Saved: output/share_dispersion_final_BC.csv")
print("Saved: output/community_breakeven_detail_BC.csv")


Saved: output/share_dispersion_final_BC.csv
Saved: output/community_breakeven_detail_BC.csv


## Outputs (sections 1-6, price/lifetime change applied, 2024 actual household counts)

All saved to `analyse_GIS_phase2_projections/output/`:

- `share_dispersion_final_BC.csv` — share_dispersion, extension capex, and TECH_HS f_min
  (GW / GWh) per cluster, using combined Source B+C household counts
- `community_breakeven_detail_BC.csv` — per-community classification (dispersed vs connectable)

See Section 7 below for the three projected-household scenarios (2024 control / 2035 / 2050).


## 7. Projected household scenarios (2024 control / 2035 / 2050)

`share_dispersion` (Section 6) fixes the household count at its 2024 value. Here the same
breakeven classification (Sections 4-5) is re-run three times, changing **only** the number of
households per cluster — everything else (2012 community geography, distances, per-HH demand
`per_hh_kwh` fixed at ~1169 kWh/HH, SHS unit costs, `MV_EUR`/`CONN_EUR`/`OM_RATE`/`I_RATE`) stays
exactly as computed above.

- **2024** — actual household counts (`scale = 1` for every cluster). Control run: since Sections
  1-6 already use the new PV/battery price and lifetime, this isolates the **price effect alone**
  from the demography effect, and is the run to compare against the published
  `analyse_GIS_phase2` values (455/242 communities, 45.58 M€, 4.84 M€/yr).
- **2035 / 2050** — projected household counts per cluster from
  `projections/output/menages_projetes.csv` (column `cluster`, summed `hh_2035`/`hh_2050`), not
  modified here.

**Scaling method**: each community's combined B+C household count (`task1_bc["HH_2024_scaled"]`,
already built in Section 5) is multiplied by a single per-cluster factor
`scale[cluster] = target_hh[cluster] / hh_2024[cluster]`. This grows every community in a cluster
proportionally — it does not redistribute households between communities or municipalities, and
it does not touch distances or per-household demand. `hh_2024` is read from the same CSV (its
`hh_2024` column matches the notebook's own `total_hh` sums per cluster exactly — both derive from
`CSV_final`'s 2024 census column), so the 2024 scenario's scale factors are exactly 1.0.


In [15]:
proj = pd.read_csv(os.path.join(BASE, "projections/output/menages_projetes.csv"))
hh_by_cluster_year = proj.groupby("cluster")[["hh_2024", "hh_2035", "hh_2050"]].sum()

hh_2024_baseline = hh_by_cluster_year["hh_2024"]

check = pd.DataFrame({
    "hh_2024 (menages_projetes.csv)": hh_2024_baseline,
    "total_hh (this notebook, Section 2)": pd.Series({c: sum(total_hh[m] for m in CLUSTERS[c]) for c in CLUSTERS}),
})
check["match"] = (check["hh_2024 (menages_projetes.csv)"] - check["total_hh (this notebook, Section 2)"]).abs() < 1e-6
assert check["match"].all(), "hh_2024 in menages_projetes.csv does not match this notebook's total_hh — check source alignment"
print("hh_2024 cluster totals from menages_projetes.csv match this notebook's total_hh exactly:")
check

SCENARIOS = {
    "2024": hh_2024_baseline,
    "2035": hh_by_cluster_year["hh_2035"],
    "2050": hh_by_cluster_year["hh_2050"],
}


hh_2024 cluster totals from menages_projetes.csv match this notebook's total_hh exactly:


In [16]:
def run_scenario(hh_scale_by_cluster):
    """Re-run the Section 4-5 breakeven classification with each community's combined B+C
    household count multiplied by a single per-cluster factor. Distances, per_hh_kwh, shs_annual,
    MV_EUR/CONN_EUR/CRF_MV are untouched — only household counts move."""
    task1_scn = task1_bc.copy()
    task1_scn["HH_2024_scaled"] = task1_scn["HH_2024_scaled"] * task1_scn["Cluster"].map(hh_scale_by_cluster)
    return classify_communities(task1_scn, shs_annual, MV_EUR, CONN_EUR, CRF_MV)


def summarize_scenario(comm, target_hh_by_cluster):
    """Cluster-level outputs: connectable/dispersed community counts, extension capex (overnight
    and annualized — CRF_MV only, matching the published headline-figure convention), dispersed
    households' annual electricity demand (GWh), and km of MT line to be connected."""
    rows = []
    for cluster in CLUSTERS:
        sub = comm[comm["Cluster"] == cluster]
        connectable = ~sub["dispersed"]
        dispersed = sub["dispersed"]
        ext_capex = (sub.loc[connectable, "distance_km"] * MV_EUR).sum() + sub.loc[connectable, "HH"].sum() * CONN_EUR
        km_mt = sub.loc[connectable, "distance_km"].sum()
        demande_dispersee_kwh = (sub.loc[dispersed, "HH"]
                                  * sub.loc[dispersed, "Municipio_std"].map(per_hh_kwh)).sum()
        rows.append({
            "Cluster": cluster,
            "N_connectable": int(connectable.sum()),
            "N_dispersed": int(dispersed.sum()),
            "N_total_communities": int(len(sub)),
            "extension_capex_EUR": round(ext_capex, 0),
            "extension_capex_annualized_EUR": round(CRF_MV * ext_capex, 0),
            "demande_dispersee_GWh": round(demande_dispersee_kwh / 1e6, 4),
            "km_MT_a_raccorder": round(km_mt, 2),
            "menages_cluster": round(target_hh_by_cluster[cluster], 0),
        })
    df = pd.DataFrame(rows)
    total = {col: df[col].sum() for col in df.columns if col != "Cluster"}
    total["Cluster"] = "TOTAL"
    for col in ["demande_dispersee_GWh"]:
        total[col] = round(total[col], 4)
    return pd.concat([df, pd.DataFrame([total])], ignore_index=True)


In [17]:
scenario_summaries = {}
scenario_communities = {}

for year, target_hh in SCENARIOS.items():
    scale = {c: target_hh[c] / hh_2024_baseline[c] for c in CLUSTERS}
    comm = run_scenario(scale)
    summary = summarize_scenario(comm, target_hh)

    year_dir = os.path.join(OUT, year)
    os.makedirs(year_dir, exist_ok=True)
    summary.to_csv(os.path.join(year_dir, "cluster_summary.csv"), index=False)
    comm.to_csv(os.path.join(year_dir, "community_detail.csv"), index=False)

    scenario_summaries[year] = summary
    scenario_communities[year] = comm
    print(f"{year}: scale factors {({c: round(v, 4) for c, v in scale.items()})} "
          f"-> saved {year_dir}/cluster_summary.csv, {year_dir}/community_detail.csv")

scenario_summaries["2024"]


2024: scale factors {'C1': np.float64(1.0), 'C2': np.float64(1.0), 'C3': np.float64(1.0), 'C4': np.float64(1.0), 'C5': np.float64(1.0)} -> saved output\2024/cluster_summary.csv, output\2024/community_detail.csv


2035: scale factors {'C1': np.float64(1.1838), 'C2': np.float64(1.2442), 'C3': np.float64(1.1901), 'C4': np.float64(1.2522), 'C5': np.float64(1.2261)} -> saved output\2035/cluster_summary.csv, output\2035/community_detail.csv
2050: scale factors {'C1': np.float64(1.3604), 'C2': np.float64(1.5013), 'C3': np.float64(1.3807), 'C4': np.float64(1.5343), 'C5': np.float64(1.4774)} -> saved output\2050/cluster_summary.csv, output\2050/community_detail.csv


,Cluster,N_connectable,N_dispersed,N_total_communities,extension_capex_EUR,extension_capex_annualized_EUR,demande_dispersee_GWh,km_MT_a_raccorder,menages_cluster
0,C1,76,88,164,9168870.0,972627.0,2.1002,1050.33,10933.0
1,C2,10,6,16,1614644.0,171280.0,0.1392,185.38,802.0
2,C3,117,43,160,7566979.0,802699.0,0.8483,815.93,40298.0
3,C4,227,116,343,19370468.0,2054805.0,1.5092,2219.59,16612.0
4,C5,14,0,14,157242.0,16680.0,0.0000,8.19,15564.0
5,TOTAL,444,253,697,37878203.0,4018091.0,4.5969,4279.42,84209.0


In [18]:
# ── 2024 (price-only-changed) vs published analyse_GIS_phase2 values ─────────────────────────
# Published (analyse_GIS_phase2/output/share_dispersion_final_BC.csv, Peña Balderrama et al. 2020
# PV/battery USD prices, PV/bat lifetimes 15/10y): 455 connectable, 242 dispersed communities,
# 45.58 M€ extension capex, 4.84 M€/yr annualized (CRF_MV x capex).
# Since the 2024 scenario keeps the actual 2024 household counts (scale = 1) and the same 2012
# community geography/distances, any difference below comes ONLY from the PV_EUR_KW / LIFE_PV /
# BAT_EUR_KWH / LIFE_BAT change made in Section 1 — not from demography.
published = {
    "N_connectable": 455,
    "N_dispersed": 242,
    "extension_capex_EUR": 45_580_000,
    "extension_capex_annualized_EUR": 4_840_000,
}
run_2024_total = scenario_summaries["2024"].set_index("Cluster").loc["TOTAL"]

comparison_2024 = pd.DataFrame([
    {"metric": k, "published": v, "this_run_2024_price_change_only": run_2024_total[k],
     "delta": run_2024_total[k] - v, "pct_change": round(100 * (run_2024_total[k] - v) / v, 2)}
    for k, v in published.items()
])
comparison_2024.to_csv(os.path.join(OUT, "2024", "comparison_vs_published.csv"), index=False)
print("2024 control run (price/lifetime change only, actual 2024 households) vs published analyse_GIS_phase2 values:")
comparison_2024


2024 control run (price/lifetime change only, actual 2024 households) vs published analyse_GIS_phase2 values:


,metric,published,this_run_2024_price_change_only,delta,pct_change
0,N_connectable,455,444.0,-11.0,-2.42
1,N_dispersed,242,253.0,11.0,4.55
2,extension_capex_EUR,45580000,37878203.0,-7701797.0,-16.90
3,extension_capex_annualized_EUR,4840000,4018091.0,-821909.0,-16.98


## Section 7 outputs

Saved to `analyse_GIS_phase2_projections/output/<year>/` for `year` in `2024`, `2035`, `2050`:

- `cluster_summary.csv` — per cluster (+ `TOTAL` row): `N_connectable`, `N_dispersed`,
  `N_total_communities`, `extension_capex_EUR` (overnight), `extension_capex_annualized_EUR`
  (`CRF_MV × capex`), `demande_dispersee_GWh` (dispersed households' annual electricity demand),
  `km_MT_a_raccorder`, and the `menages_cluster` household count used for that scenario.
- `community_detail.csv` — per-community classification (dispersed vs connectable) for that
  scenario, same structure as `communities_bc` in Section 4.

Additionally, `output/2024/comparison_vs_published.csv` — the 2024 (price-only-changed) totals
against the published `analyse_GIS_phase2` figures, to isolate the price effect from the
demography effect. No interpretation beyond these figures is drawn here.
